In [ ]:
import math
 
# ══════════════════════════════════════════════════════════════════
# Tiện ích ma trận
# ══════════════════════════════════════════════════════════════════
 
def mz(m, n): return [[0.]*n for _ in range(m)]
def mI(n):    I=mz(n,n); [I[i].__setitem__(i,1.) for i in range(n)]; return I
def mc(A):    return [r[:] for r in A]
def mT(A):
    m,n=len(A),len(A[0]); B=mz(n,m)
    for i in range(m):
        for j in range(n): B[j][i]=A[i][j]
    return B
def mm(A,B):
    m,k,n=len(A),len(B),len(B[0]); C=mz(m,n)
    for i in range(m):
        for p in range(k):
            if A[i][p]==0.: continue
            for j in range(n): C[i][j]+=A[i][p]*B[p][j]
    return C
 
# ══════════════════════════════════════════════════════════════════
# PHẦN 2 — Phase 1: Golub-Kahan Bidiagonalization
#          Cải tiến: Deferred U/V accumulation
#
# Thay vì cập nhật U, V NGAY BÊN TRONG vòng lặp bidiag, ta chỉ
# lưu (v, beta) của từng Householder.  khi bidiag hoàn tất,
# xây U và V qua backward accumulation (đi ngược từ k=p-1 về 0,
# nhân từng Householder từ TRÁI).  Việc này:
#   • Giữ vòng lặp chính của bidiag gọn — chỉ biến đổi ma trận A
#   • Tránh truy cập bộ nhớ phân mảnh xen kẽ giữa A và U/V
#   • Python list cache thân thiện hơn khi các thao tác batch lại
# ══════════════════════════════════════════════════════════════════
 
def _hh(x):
    """Householder: trả về (v, beta) với H = I - beta*v*vᵀ, v[0]=1."""
    n=len(x); sig=sum(x[i]**2 for i in range(1,n)); v=x[:]
    if sig<1e-300 and x[0]>=0: return v, 0.
    r=math.sqrt(x[0]**2+sig)
    v[0]=x[0]-r if x[0]<=0 else -sig/(x[0]+r)
    beta=2*v[0]**2/(sig+v[0]**2); sc=v[0]
    v=[vi/sc for vi in v]
    return v, beta
 
def _apL(A, v, b, rs, cs):
    """A[rs:, cs:] ← H_v A[rs:, cs:]  (apply left Householder)"""
    n=len(A[0]); k=len(v)
    for j in range(cs, n):
        d=sum(v[i]*A[rs+i][j] for i in range(k))
        for i in range(k): A[rs+i][j] -= b*v[i]*d
 
def _apR(A, v, b, rs, cs):
    """A[rs:, cs:] ← A[rs:, cs:] H_v  (apply right Householder)"""
    m=len(A); k=len(v)
    for i in range(rs, m):
        d=sum(A[i][cs+j]*v[j] for j in range(k))
        for j in range(k): A[i][cs+j] -= b*v[j]*d
 
def bidiag(A_in):
    """
    A (m×n) → (U0, B, V0)  s.t.  B = U0 · A · V0,  B upper bidiagonal.
    U0 ∈ ℝᵐˣᵐ, V0 ∈ ℝⁿˣⁿ đều trực giao.
 
    Kỹ thuật: lưu Householder vectors; backward accumulation sau vòng lặp.
    """
    A=mc(A_in); m,n=len(A),len(A[0]); p=min(m,n)
    lvecs=[]   # [(v_l, beta_l), ...]  – Householder trái
    rvecs=[]   # [(v_r, beta_r), ...]  – Householder phải
 
    # ── Vòng lặp chính: chỉ biến đổi A, lưu vectors ──────────────
    for k in range(p):
        col=[A[i][k] for i in range(k,m)]
        vl,bl=_hh(col); lvecs.append((vl,bl))
        if bl: _apL(A, vl, bl, k, k)
 
        if k+1<n:
            row=[A[k][j] for j in range(k+1,n)]
            vr,br=_hh(row); rvecs.append((vr,br))
            if br: _apR(A, vr, br, k, k+1)
        else:
            rvecs.append((None, 0.))
 
    # ── Backward accumulation: U = H₀·H₁·…·H_{p-1} ──────────────
    # Q ← H_k·Q tác động lên hàng k..m-1 của Q.
    # Tại bước k (đi ngược), các cột j < k của Q vẫn là e_j (identity —
    # chưa bị H_{k+1}..H_{p-1} chạm tới).  Dot-product với v_k (có
    # v_k[0]=1 ở hàng k, còn lại ở hàng k+1..) trên cột j < k luôn = 0.
    # → Chỉ cần duyệt j = k..m-1, bỏ qua toàn bộ vùng identity bên trái.
    U=mI(m)
    for k in range(p-1, -1, -1):
        vl, bl = lvecs[k]
        if bl:
            for j in range(k, m):         # chỉ j ≥ k — vùng j<k là zero
                d=sum(vl[i]*U[k+i][j] for i in range(len(vl)))
                for i in range(len(vl)): U[k+i][j]-=bl*vl[i]*d
 
    # ── Backward accumulation: V = G₀·G₁·…·G_{r-1} ───────────────
    # Tương tự: H_k^{right} tại bước k tác động từ cột st=k+1 trở đi.
    # Các cột j < st vẫn là e_j → chỉ duyệt j = st..n-1.
    V=mI(n)
    for k in range(len(rvecs)-1, -1, -1):
        vr, br = rvecs[k]
        if vr is not None and br:
            st=k+1
            for j in range(st, n):        # chỉ j ≥ st — vùng j<st là zero
                d=sum(vr[i]*V[st+i][j] for i in range(len(vr)))
                for i in range(len(vr)): V[st+i][j]-=br*vr[i]*d
 
    # B = U · A_orig · V  (xem debug bên dưới tại sao là U, không U^T)
    return U, A, V
 
 
# ══════════════════════════════════════════════════════════════════
# PHẦN 3 — Phase 2: Golub-Reinsch QR iteration
#          Cải tiến 1: Scaled Wilkinson shift
#          Cải tiến 2: Bulge-chasing sạch hơn
# ══════════════════════════════════════════════════════════════════
 
def _gv(a, b):
    """(c, s) s.t.  [[c,s],[-s,c]] · [a;b] = [r;0]."""
    r=math.hypot(a, b)
    return (1., 0.) if r<1e-300 else (a/r, b/r)
 
def _rcols(Q, j1, j2, c, s):
    """Q → Q · [[c,-s],[s,c]]  trên cột j1, j2."""
    for i in range(len(Q)):
        a,b=Q[i][j1],Q[i][j2]
        Q[i][j1]=c*a+s*b; Q[i][j2]=-s*a+c*b
 
def _qr_step(d, e, ql, qr, U, V):
    """
    Một Golub-Kahan QR sweep (implicit Wilkinson shift) trên block [ql..qr].
    d, e cập nhật in-place.  U, V (p×p) tích lũy phép xoay.
 
    ── Wilkinson shift có scaling ──────────────────────────────────
    Trailing 2×2 của BᵀB:
        T = [[dm1²+em1²,  dm1·em],
             [dm1·em,     dm²+em²]]
    Nếu tính trực tiếp, dm**2 tràn số khi |dm| > ~1e154 (float64).
    Giải pháp: chia hết các phần tử cho  scale = max(|dm1|,|em1|,|dm|,|em|)
    trước khi bình phương, rồi nhân ngược lại sau:
        mu_scaled = eigenvalue_gần_t22 của T/scale²
        mu = scale² · mu_scaled
 
    ── Bulge-chasing và cập nhật e[k-1] ───────────────────────────
    Tại bước k > ql, phép xoay PHẢI G_k tác động lên cột k và k+1:
      • Hàng k-1:  [e[k-1], fill]  →  [r, 0]   (r = hypot(e[k-1], fill))
      • Hàng k:    [d[k], e[k]]    →  cập nhật theo c,s
      • Hàng k+1:  [0, d[k+1]]    →  [bulge, nd1R]
    Cập nhật e[k-1] = r là hệ quả TRỰC TIẾP của phép xoay phải;
    """
    nb = qr - ql + 1
 
    # ── Scaled Wilkinson shift ──────────────────────────────────
    dm  = d[qr]
    dm1 = d[qr-1] if nb>1 else 0.
    em  = e[qr-1] if nb>1 else 0.
    em1 = e[qr-2] if nb>2 else 0.
 
    scale = max(abs(dm1), abs(em1), abs(dm), abs(em))
    if scale < 1e-300:
        # Toàn bộ trailing block xấp xỉ zero → deflate e[qr-1] trực tiếp.
        # Không được return mà không đổi trạng thái: vòng lặp ngoài sẽ
        # không deflate được và rơi vào infinite loop (500*p lần vô ích).  ## hạn chế còn lại
        e[qr-1] = 0.
        return
    dm1s=dm1/scale; em1s=em1/scale; dms=dm/scale; ems=em/scale
 
    t11 = dm1s**2 + em1s**2
    t22 = dms**2  + ems**2
    t12 = dm1s * ems
    dlt = (t11 - t22) / 2.
    disc = math.sqrt(dlt**2 + t12**2) if dlt**2+t12**2>0 else 0.
    den  = dlt + math.copysign(disc, dlt)
    mu   = scale**2 * (t22 - (t12**2/den if abs(den)>1e-300 else 0.))
 
    # ── Global scaling cho f, g (Fix 3) ────────────────────────────
    # d[ql]**2 overflow nếu |d[ql]| ≈ 1e160.  Giải pháp: tính f và g
    # trong không gian đã scale bởi gscale = max(|d[i]|, |e[i]|) toàn block.
    # (c, s) của Givens chỉ phụ thuộc tỷ lệ f/g → scale đồng đều không đổi c,s.
    # mu cũng cần scale: mu_s = mu / gscale**2,  f_s = (d[ql]/gscale)**2 - mu_s
    gscale = max(
        max(abs(d[i]) for i in range(ql, qr+1)),
        max((abs(e[i]) for i in range(ql, qr)), default=0.)
    )
    if gscale < 1e-300:
        e[qr-1] = 0.; return
    dql_s = d[ql] / gscale
    eql_s = e[ql] / gscale
    mu_s  = mu / gscale**2          # mu đã được scale xuống cùng đơn vị
    f = dql_s**2 - mu_s             # an toàn: |dql_s| ≤ 1 theo định nghĩa gscale
    g = dql_s * eql_s               # an toàn tương tự
 
    for k in range(ql, qr):
        # 1. Phép xoay PHẢI G_k: [f, g] → [r, 0]
        c, s = _gv(f, g)
        _rcols(V, k, k+1, c, s)
 
        # Hệ quả của G_k trên hàng k-1 của B (khi k > ql):
        #   B[k-1][k]   = c·f + s·g  (= r = hypot(f,g), biểu diễn dạng dot-product)
        #   B[k-1][k+1] = -s·f + c·g = 0   (fill bị triệt tiêu)
        if k > ql:
            e[k-1] = c*f + s*g      # dot-product của hàng rotation với [f,g]
 
        # Hệ quả của G_k trên hàng k và k+1 (từ bidiagonal structure):
        od, oe, od1 = d[k], e[k], d[k+1]
        ndkR  = c*od  + s*oe        # B[k][k]   sau xoay phải
        nekR  = -s*od + c*oe        # B[k][k+1] sau xoay phải
        bulge = s*od1               # fill tại B[k+1][k]
        nd1R  = c*od1               # B[k+1][k+1] sau xoay phải
 
        # 2. Phép xoay TRÁI H_k: triệt tiêu bulge tại (k+1, k)
        c2, s2 = _gv(ndkR, bulge)
        _rcols(U, k, k+1, c2, s2)
 
        # Cập nhật B sau H_k (tác động lên cột k và k+1, hàng k và k+1):
        d[k]   = c2*ndkR + s2*bulge     # = hypot(ndkR, bulge)
        nek    = c2*nekR  + s2*nd1R
        d[k+1] = -s2*nekR + c2*nd1R
        e[k]   = nek
 
        # 3. Propagate: fill mới tại (k, k+2) từ phép xoay trái
        if k+1 < qr:
            fill   = s2 * e[k+1]
            e[k+1] = c2 * e[k+1]
            f, g   = nek, fill
 
 
def bidiag_svd(d_in, e_in, p):
    """
    SVD của bidiagonal p×p: B = U diag(σ) Vᵀ
    d: diagonal, e: superdiagonal.  Trả về (U, sigma, V).
    """
    d=d_in[:p][:]; e=(e_in[:p-1] if p>1 else [])[:]
    U=mI(p); V=mI(p)
    eps=1e-13
    for _ in range(500*p):
        for i in range(p-1):
            if abs(e[i])<=eps*(abs(d[i])+abs(d[i+1])): e[i]=0.
        qr=p-1
        while qr>0 and e[qr-1]==0.: qr-=1
        if qr==0: break
        ql=qr-1
        while ql>0 and e[ql-1]!=0.: ql-=1
        _qr_step(d, e, ql, qr, U, V)
    for i in range(p):
        if d[i]<0.:
            d[i]=-d[i]
            for r in range(p): U[r][i]=-U[r][i]
    idx=sorted(range(p), key=lambda i:-d[i])
    sig=[d[i] for i in idx]
    Us=[[U[r][idx[c]] for c in range(p)] for r in range(p)]
    Vs=[[V[r][idx[c]] for c in range(p)] for r in range(p)]
    return Us, sig, Vs
 
 
# ══════════════════════════════════════════════════════════════════
# PHẦN 4 — Full SVD
# ══════════════════════════════════════════════════════════════════
 
def svd(A_in):
    """Full SVD: A = U diag(sigma) Vᵀ,  sigma giảm dần."""
    m, n = len(A_in), len(A_in[0])
    if m < n:                       # wide: transpose trick
        V, sigma, U = svd(mT(A_in))
        return U, sigma, V
    p = min(m, n)
    # bidiag() trả về (U0, B, V0) s.t. B = U0ᵀ · A · V0
    # → A = U0 · B · V0ᵀ,  U_final = U0·Ul,  V_final = V0·Vr
    U0, B, V0 = bidiag(A_in)
    d=[B[i][i] for i in range(p)]; e=[B[i][i+1] for i in range(p-1)]
    Ul, sigma, Vr = bidiag_svd(d, e, p)
    Uf=mI(m); Vf=mI(n)
    for i in range(p):
        for j in range(p): Uf[i][j]=Ul[i][j]; Vf[i][j]=Vr[i][j]
    return mm(U0, Uf), sigma, mm(V0, Vf)
 